In [ ]:
# Install required packages (uncomment if needed)
# !pip install transformers datasets accelerate peft bitsandbytes torch sentencepiece

In [ ]:
import torch
import json
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Configuration

In [ ]:
# ============= CONFIGURATION =============
# Change these settings based on your needs

# Subject to train on: "dl" (Deep Learning), "da" (Data Analytics), "ime" (IME)
SUBJECT = "dl"

# Base model - TinyLlama works well for CPU/limited GPU
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Training parameters
NUM_EPOCHS = 3
BATCH_SIZE = 2  # Reduce if out of memory
LEARNING_RATE = 2e-4
MAX_LENGTH = 512

# LoRA parameters
LORA_R = 16  # Rank
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Paths
DATA_DIR = Path("../data")
BACKEND_DATA_DIR = Path("../backend/data")
MODEL_OUTPUT_DIR = Path(f"../models/{SUBJECT}_personalized_lora")

print(f"Training model for subject: {SUBJECT.upper()}")
print(f"Output directory: {MODEL_OUTPUT_DIR}")

## Load and Prepare Training Data

In [ ]:
def load_training_data(subject: str) -> list:
    """Load training data from JSON file"""
    # Try different data locations
    possible_paths = [
        DATA_DIR / f"training_data_{subject}.json",
        BACKEND_DATA_DIR / f"training_data_{subject}.json",
        DATA_DIR / f"extracted_text_{subject}.json",
        BACKEND_DATA_DIR / f"extracted_text_{subject}.json"
    ]
    
    for path in possible_paths:
        if path.exists():
            print(f"Loading data from: {path}")
            with open(path, 'r', encoding='utf-8') as f:
                return json.load(f)
    
    raise FileNotFoundError(f"No training data found for subject: {subject}")

def prepare_training_examples(data: list) -> list:
    """Convert raw data to instruction-following format for question generation"""
    training_examples = []
    
    # Question generation templates
    templates = [
        {
            "instruction": "Generate exam questions based on the following content.",
            "task": "question_generation"
        },
        {
            "instruction": "Create multiple choice questions from this material.",
            "task": "mcq_generation"
        },
        {
            "instruction": "Generate short answer questions from the content below.",
            "task": "short_answer"
        },
        {
            "instruction": "Create essay questions based on this academic content.",
            "task": "essay_questions"
        }
    ]
    
    for i, item in enumerate(data):
        # Get content from either format
        if isinstance(item, dict):
            if 'output' in item:
                content = item['output']
            elif 'content' in item:
                content = item['content']
            else:
                continue
        else:
            content = str(item)
        
        # Skip empty or very short content
        if not content or len(content.strip()) < 100:
            continue
        
        # Create training examples with different templates
        template = templates[i % len(templates)]
        
        training_examples.append({
            "instruction": template["instruction"],
            "input": content[:1500],  # Limit input length
            "output": f"Based on the provided content, here are relevant exam questions:\n\n{content[:500]}"
        })
    
    return training_examples

# Load data
raw_data = load_training_data(SUBJECT)
training_data = prepare_training_examples(raw_data)

print(f"\nLoaded {len(raw_data)} raw examples")
print(f"Created {len(training_data)} training examples")
print(f"\nSample training example:")
print(json.dumps(training_data[0], indent=2, ensure_ascii=False)[:500])

## Load Base Model and Tokenizer

In [ ]:
print(f"Loading base model: {BASE_MODEL}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model
if device == "cuda":
    # For GPU with limited memory, use 8-bit quantization
    try:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            torch_dtype=torch.float16,
            device_map="auto",
            load_in_8bit=True
        )
        print("Loaded model with 8-bit quantization")
    except Exception as e:
        print(f"8-bit loading failed, using float16: {e}")
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            torch_dtype=torch.float16,
            device_map="auto"
        )
else:
    # CPU mode
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )

print(f"Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

## Configure LoRA

In [ ]:
# Configure LoRA for efficient fine-tuning
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Target attention layers
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

print("\nLoRA Configuration:")
print(f"  Rank (r): {LORA_R}")
print(f"  Alpha: {LORA_ALPHA}")
print(f"  Dropout: {LORA_DROPOUT}")
print("\nTrainable Parameters:")
model.print_trainable_parameters()

## Prepare Dataset

In [ ]:
def format_prompt(example: dict) -> str:
    """Format example into instruction-following prompt"""
    prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    return prompt

def tokenize_function(examples):
    """Tokenize training examples"""
    texts = [format_prompt(ex) for ex in [examples]]
    
    result = tokenizer(
        format_prompt(examples),
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None
    )
    
    # For causal LM, labels = input_ids
    result["labels"] = result["input_ids"].copy()
    
    return result

# Create dataset
dataset = Dataset.from_list(training_data)

# Tokenize
tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=dataset.column_names,
    desc="Tokenizing"
)

# Split into train/eval
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"\nDataset Statistics:")
print(f"  Training examples: {len(train_dataset)}")
print(f"  Evaluation examples: {len(eval_dataset)}")
print(f"  Max sequence length: {MAX_LENGTH}")

## Training

In [ ]:
# Create output directory
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training arguments
training_args = TrainingArguments(
    output_dir=str(MODEL_OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",  # Disable wandb
    fp16=(device == "cuda"),
    dataloader_pin_memory=False,
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("\nTraining Configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Output: {MODEL_OUTPUT_DIR}")
print("\nStarting training...")

In [ ]:
# Train the model
train_result = trainer.train()

print("\n" + "="*50)
print("TRAINING COMPLETE!")
print("="*50)
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")

## Save Model

In [ ]:
# Save the final model
final_model_path = MODEL_OUTPUT_DIR / "final"
final_model_path.mkdir(parents=True, exist_ok=True)

# Save LoRA adapter
model.save_pretrained(str(final_model_path))
tokenizer.save_pretrained(str(final_model_path))

print(f"\nModel saved to: {final_model_path}")
print("\nSaved files:")
for f in final_model_path.iterdir():
    print(f"  - {f.name}")

## Test the Model

In [ ]:
def generate_questions(prompt: str, max_new_tokens: int = 200) -> str:
    """Generate questions using the fine-tuned model"""
    formatted_prompt = f"""### Instruction:
Generate exam questions based on the following content.

### Input:
{prompt}

### Response:
"""
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    
    if device == "cuda":
        inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the response part
    if "### Response:" in response:
        response = response.split("### Response:")[-1].strip()
    
    return response

# Test with sample content
test_content = """
Deep learning is a subset of machine learning that uses neural networks with many layers.
Key concepts include backpropagation, gradient descent, and activation functions.
Popular architectures include CNNs for image processing and RNNs for sequence data.
"""

print("Testing model with sample content...")
print("="*50)
print("Input:", test_content[:100], "...")
print("="*50)
print("\nGenerated Questions:")
print(generate_questions(test_content))

## Integration with Backend

To use this model in the backend, copy the model to the backend models directory:

```bash
# Copy model files
cp -r ../models/{subject}_personalized_lora/final ../backend/models/{subject}_personalized/
```

Then update `backend/utils/gemini_service.py` to optionally use the local model for personalized generation.

In [ ]:
# Summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Subject: {SUBJECT.upper()}")
print(f"Base Model: {BASE_MODEL}")
print(f"Training Examples: {len(train_dataset)}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Final Model: {final_model_path}")
print("\nNext Steps:")
print("1. Test the model with more examples")
print("2. Adjust hyperparameters if needed")
print("3. Integrate with the backend API")
print("="*60)